<a id="ksc2026-start"></a>
# 00 Start Here · 5분 환경 점검

이 노트북은 **실습을 시작하기 전에 실행 환경만 확인하는 첫 화면**입니다. 개념 설명과 상세 실습 지도는 [전체 과정 안내](README.md), [GH200 모듈 지도](01_GH200/README.md), [PhysicsNeMo 모듈 지도](02_PhysicsNeMo/README.md)에서 확인합니다.

## 실행 방법

1. JupyterLab 메뉴에서 **Run → Run All Cells**를 선택합니다.
2. 맨 아래에 표시되는 네 개의 `READY`를 확인합니다. 약 5분이 걸리며 학습은 아직 시작하지 않습니다.
3. 첫 번째 실습인 [Grace CPU 컴파일·튜닝](01_GH200/01_CPU_Compile_and_Tune.ipynb)을 엽니다.

브라우저는 참가자 컴퓨터에서 열리지만, 이 노트북의 Python과 GPU 계산은 Slurm이 동적으로 배정한 GH200 한 개에서 실행됩니다. 참가자에게는 GH200 한 개가 보이며, 노트북 안에서는 `cuda:0`으로 표시됩니다.

## 오늘 수행할 네 가지 실습

| 순서 | 실습 | 직접 확인할 결과 | 완료 기준 |
|---:|---|---|---|
| 1 | [Grace CPU 컴파일·튜닝](01_GH200/01_CPU_Compile_and_Tune.ipynb) | OpenBLAS·NVPL 실행 파일과 성능표 | 두 실행의 계산 결과를 확인하고 가장 빠른 조건을 설명 |
| 2 | [Hopper GPU 메모리·프로파일링](01_GH200/02_GPU_Memory_Profile.ipynb) | 세 메모리 방식의 실행 결과와 Nsight 보고서 | 지원되는 경로의 결과가 일치하고 실행 순서의 차이를 설명 |
| 3 | [발사체 운동 PINN](02_PhysicsNeMo/01_Projectile_PINN.ipynb) | 예측 궤적, 검증 오차, 저장된 체크포인트 | 초기조건·운동방정식이 손실에 들어가는 위치와 예측 오차를 확인 |
| 4 | [Poisson FNO](02_PhysicsNeMo/02_Poisson_FNO.ipynb) | 테스트 오차, 예측 그림, 실행 시간·GPU 메모리 | 처음 보는 소스항에 대한 예측 결과와 계산 비용을 해석 |

필수 실습을 일찍 마치면 강사 안내에 따라 [FNO 푸리에 모드 수 비교](02_PhysicsNeMo/optional/FNO_Mode_Ablation.ipynb)를 진행합니다.

## 저장과 재접속

- Jupyter는 60초마다 자동 저장합니다. 중요한 변경 뒤에는 macOS에서 `Cmd+S`, Windows·Linux에서 `Ctrl+S`를 누릅니다.
- 저장 파일은 개인 `/scratch/<로그인계정>/ksc2026/workspaces/`에 남습니다. SSH 터널이나 Slurm Job이 끊겨도 삭제되지 않습니다.
- 다시 접속할 때는 PILOT 로그인 노드에서 `/scratch/hackathon/ksc2026/bin/ksc2026`을 실행하고, 화면에 표시된 새 터널 명령과 주소를 사용합니다.
- 계산 노드에서는 `apt`, `pip`, `git`, `wget`, `curl`을 실행하지 않습니다. 실습에 필요한 도구와 Python 패키지는 교육용 SIF에 포함되어 있습니다.


## 1. 과정 폴더 확인

현재 위치에서 과정의 최상위 폴더와 실습 지원 파일의 위치를 찾습니다.


In [ ]:
from pathlib import Path

launch_dir = Path.cwd().resolve()

def is_course_root(path):
    return (
        (path / "00_Start_Here.ipynb").is_file()
        and (path / "labs" / "gh200").is_dir()
        and (path / "labs" / "projectile").is_dir()
        and (path / "labs" / "poisson_fno").is_dir()
    )

REPO_ROOT = next(
    (candidate for candidate in (launch_dir, *launch_dir.parents) if is_course_root(candidate)),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError(
        "KSC 2026 과정의 최상위 폴더를 찾지 못했습니다. 참가자 작업공간에서 "
        "00_Start_Here.ipynb를 열었는지 확인하세요."
    )

GH200_DIR = REPO_ROOT / "labs" / "gh200"
PROJECTILE_DIR = REPO_ROOT / "labs" / "projectile"
FNO_DIR = REPO_ROOT / "labs" / "poisson_fno"

print(f"Launch directory : {launch_dir}")
print(f"Course root      : {REPO_ROOT}")
print(f"GH200 lab        : {GH200_DIR}")
print(f"Projectile lab   : {PROJECTILE_DIR}")
print(f"Poisson FNO lab  : {FNO_DIR}")


## 2. 네 가지 실습의 필수 파일 확인

번호가 붙은 노트북과 각 노트북이 호출하는 소스 코드·설정 파일을 확인합니다.


In [ ]:
session_files = {
    "01-1 Grace CPU": [
        "01_GH200/01_CPU_Compile_and_Tune.ipynb",
        "labs/gh200/notebook_utils.py",
        "labs/gh200/blas/Makefile",
        "labs/gh200/blas/dgemm.c",
    ],
    "01-2 Hopper GPU": [
        "01_GH200/02_GPU_Memory_Profile.ipynb",
        "labs/gh200/notebook_utils.py",
        "labs/gh200/cuda_memory/explicit.cu",
        "labs/gh200/cuda_memory/managed.cu",
        "labs/gh200/cuda_memory/hmm.cu",
    ],
    "02-1 발사체 운동 PINN": [
        "02_PhysicsNeMo/01_Projectile_PINN.ipynb",
        "labs/projectile/source_code/projectile.py",
        "labs/projectile/source_code/projectile_eqn.py",
        "labs/projectile/source_code/conf/config.yaml",
        "labs/projectile/images/projectile.svg",
        "labs/projectile/images/physicsnemo_sym_workflow.webp",
    ],
    "02-2 Poisson FNO": [
        "02_PhysicsNeMo/02_Poisson_FNO.ipynb",
        "labs/poisson_fno/data_validation.py",
        "labs/poisson_fno/generate_data.py",
        "labs/poisson_fno/train_fno.py",
        "labs/poisson_fno/notebook_utils.py",
        "labs/poisson_fno/images/fno_data_flow.svg",
        "labs/poisson_fno/conf/config_FNO.yaml",
        "labs/poisson_fno/conf/config_FNO_recovery.yaml",
    ],
}

SESSION_FILES_READY = {}
for session, relative_paths in session_files.items():
    print(f"\n[{session}]")
    checks = []
    for relative in relative_paths:
        exists = (REPO_ROOT / relative).is_file()
        checks.append(exists)
        print(f"{'PASS' if exists else 'FAIL':4s}  {relative}")
    SESSION_FILES_READY[session] = all(checks)

optional_paths = [
    REPO_ROOT / "02_PhysicsNeMo" / "optional" / "README.md",
    REPO_ROOT / "02_PhysicsNeMo" / "optional" / "FNO_Mode_Ablation.ipynb",
]
print("\n[PhysicsNeMo optional]")
for optional_path in optional_paths:
    print(f"{'PASS' if optional_path.is_file() else 'WARN':4s}  {optional_path.relative_to(REPO_ROOT)}")


## 3. ARM64 도구와 교육용 SIF 구성 확인

Grace CPU·Hopper GPU 실습에 필요한 명령과 교육용 컨테이너의 버전 정보를 확인합니다. 이 단계에서는 설치나 다운로드를 실행하지 않습니다.


In [ ]:
import json
import platform
import shutil

architecture = platform.machine().lower()
ARCH_READY = architecture in {"aarch64", "arm64"}
print(f"{'PASS' if ARCH_READY else 'FAIL'}  Architecture     {architecture} (expected aarch64/arm64)")

tool_names = ("gcc", "nvc", "nvcc", "nsys", "nvbandwidth", "make")
TOOL_PATHS = {name: shutil.which(name) for name in tool_names}
for name, path in TOOL_PATHS.items():
    print(f"{'PASS' if path else 'FAIL':4s}  {name:16s} {path or 'not found'}")
TOOLS_READY = all(TOOL_PATHS.values())

manifest_candidates = [
    Path("/etc/ksc2026-image.json"),
    REPO_ROOT / "container" / "ksc2026-image.json",
]
manifest_path = next((path for path in manifest_candidates if path.is_file()), None)
IMAGE_MANIFEST = None
if manifest_path is None:
    print("FAIL  Image manifest   /etc/ksc2026-image.json not found")
    MANIFEST_READY = False
else:
    IMAGE_MANIFEST = json.loads(manifest_path.read_text(encoding="utf-8"))
    manifest_text = json.dumps(IMAGE_MANIFEST, sort_keys=True)
    required_versions = ("25.11", "25.5", "0.3.31", "0.8")
    missing_versions = [value for value in required_versions if value not in manifest_text]
    MANIFEST_READY = not missing_versions
    print(f"{'PASS' if MANIFEST_READY else 'FAIL'}  Image manifest   {manifest_path}")
    if missing_versions:
        print(f"      Missing version markers: {missing_versions}")
    print(json.dumps(IMAGE_MANIFEST, indent=2, ensure_ascii=False))


## 4. PhysicsNeMo와 GH200 확인

### 4-1. `nvidia-smi`로 배정된 GPU 확인

아래 셀은 Jupyter 커널이 실행 중인 Slurm 계산 노드의 GPU를 보여 줍니다. Jupyter 코드 셀에서 Linux 명령을 실행할 때는 명령 앞에 `!`를 붙이고, 로그인 터미널에서는 느낌표 없이 `nvidia-smi`를 입력합니다.

표에서 GPU 모델, 드라이버 버전, GPU 메모리 사용량을 확인합니다. 표 상단의 `CUDA Version`은 드라이버가 지원하는 CUDA 호환 수준이며, 교육용 SIF 안의 `nvcc`·PyTorch 버전과 같을 필요는 없습니다.


In [ ]:
!nvidia-smi


### 4-2. Python 패키지와 CUDA 연산 확인

교육용 SIF에 포함된 Python 패키지를 불러오고, PyTorch가 배정된 GH200에서 실제 텐서 연산을 수행하는지 확인합니다. 참가자 세션에는 GPU 한 개가 배정되고, 강사 세션에는 여러 GPU가 보일 수 있습니다. 런처 또는 Slurm의 배정 수가 컨테이너에 전달되면 정확히 일치하는지 확인합니다. 해당 정보가 없는 세션에서는 참가자용 한 개 또는 강사용 네 개가 보이는지 확인합니다. 드라이버는 R570 이상이어야 하며, 실제 CUDA 텐서 연산까지 통과해야 최종 정상으로 판정합니다. GPU 메모리 용량은 장치 정보로만 표시하며 합격 여부에는 사용하지 않습니다.


In [ ]:
import importlib
import os
import re
import shutil
import subprocess
import sys

modules_to_check = [
    ("torch", "PyTorch"),
    ("physicsnemo", "PhysicsNeMo"),
    ("physicsnemo.sym", "PhysicsNeMo-Sym"),
    ("h5py", "HDF5"),
    ("hydra", "Hydra"),
    ("matplotlib", "Matplotlib"),
]
imported = {}
import_errors = {}

print(f"Python           {sys.version.split()[0]}")
for module_name, display_name in modules_to_check:
    try:
        module = importlib.import_module(module_name)
        imported[module_name] = module
        version = getattr(module, "__version__", "version unavailable")
        print(f"PASS  {display_name:16s} {version}")
    except Exception as exc:
        import_errors[module_name] = f"{type(exc).__name__}: {exc}"
        print(f"FAIL  {display_name:16s} {import_errors[module_name]}")

STACK_READY = not import_errors
torch = imported.get("torch")
CUDA_READY = bool(torch is not None and torch.cuda.is_available())
GH200_READY = False
SM90_READY = False
GPU_COUNT_READY = False
CUDA_TENSOR_READY = False
DRIVER_READY = False

def parse_gpu_count(value):
    match = re.search(r"(?:^|:)([1-9][0-9]*)(?:\(.*\))?$", value or "")
    return int(match.group(1)) if match else None

configured_gpu_count = parse_gpu_count(os.environ.get("KSC_EXPECTED_GPU_COUNT"))
slurm_gpu_count = parse_gpu_count(os.environ.get("SLURM_GPUS_ON_NODE"))
EXPECTED_GPU_COUNT = configured_gpu_count or slurm_gpu_count

visible_gpu_count = torch.cuda.device_count() if CUDA_READY else 0
if EXPECTED_GPU_COUNT is None:
    GPU_COUNT_READY = visible_gpu_count in {1, 4}
    expected_gpu_label = "1 (참가자) 또는 4 (강사)"
else:
    GPU_COUNT_READY = visible_gpu_count == EXPECTED_GPU_COUNT
    expected_gpu_label = str(EXPECTED_GPU_COUNT)
print(
    f"{'PASS' if GPU_COUNT_READY else 'FAIL'}  Visible GPUs     "
    f"{visible_gpu_count} (expected {expected_gpu_label})"
)

if CUDA_READY:
    gpu_name = torch.cuda.get_device_name(0)
    properties = torch.cuda.get_device_properties(0)
    GH200_READY = "GH200" in gpu_name.upper()
    SM90_READY = (properties.major, properties.minor) == (9, 0)
    print(f"PASS  CUDA             {torch.version.cuda}")
    print(f"{'PASS' if GH200_READY else 'FAIL'}  GPU              {gpu_name}")
    print(
        f"{'PASS' if SM90_READY else 'FAIL'}  Compute capability "
        f"{properties.major}.{properties.minor} (expected 9.0)"
    )
    print(f"INFO  GPU memory       {properties.total_memory / 2**30:.1f} GiB")
    try:
        for gpu_index in range(visible_gpu_count):
            with torch.cuda.device(gpu_index):
                probe = torch.arange(
                    1024, device=f"cuda:{gpu_index}", dtype=torch.float32
                )
                if float((probe * 2).sum().item()) != 1047552.0:
                    raise RuntimeError(f"GPU {gpu_index} tensor result mismatch")
                torch.cuda.synchronize(gpu_index)
        CUDA_TENSOR_READY = GPU_COUNT_READY
        print(
            f"{'PASS' if CUDA_TENSOR_READY else 'FAIL'}  CUDA tensor op    "
            f"{visible_gpu_count} visible GPU(s) synchronized"
        )
    except Exception as exc:
        CUDA_TENSOR_READY = False
        print(f"FAIL  CUDA tensor op    {type(exc).__name__}: {exc}")
else:
    print("FAIL  CUDA GPU를 사용할 수 없습니다.")

if shutil.which("nvidia-smi"):
    query = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
        capture_output=True,
        text=True,
        check=False,
    )
    detail = query.stdout.strip() or query.stderr.strip()
    NVIDIA_SMI_COMMAND_READY = query.returncode == 0
    print(f"{'PASS' if NVIDIA_SMI_COMMAND_READY else 'FAIL'}  nvidia-smi       {detail}")

    driver_query = subprocess.run(
        ["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader,nounits"],
        capture_output=True,
        text=True,
        check=False,
    )
    versions = [line.strip() for line in driver_query.stdout.splitlines() if line.strip()]
    try:
        majors = [int(version.split(".", 1)[0]) for version in versions]
    except ValueError:
        majors = []
    DRIVER_READY = (
        driver_query.returncode == 0
        and len(majors) == visible_gpu_count
        and bool(majors)
        and all(major >= 570 for major in majors)
    )
    unique_versions = list(dict.fromkeys(versions))
    if len(unique_versions) == 1 and len(versions) > 1:
        driver_detail = f"{unique_versions[0]} ({len(versions)} visible GPUs)"
    else:
        driver_detail = ", ".join(unique_versions) if unique_versions else "unavailable"
    print(
        f"{'PASS' if DRIVER_READY else 'FAIL'}  NVIDIA driver    "
        f"{driver_detail} "
        "(R570 이상 필요; R570은 이미지의 CUDA forward compatibility 사용)"
    )
else:
    NVIDIA_SMI_COMMAND_READY = False
    print("FAIL  nvidia-smi       not found")


## 5. 참가자 작업 폴더 확인

학습 결과와 프로파일링 보고서는 지원 소스 코드 옆의 `work/` 또는 `outputs/` 폴더에 저장됩니다. 이 셀은 쓰기 권한을 확인하기 위해 작은 임시 파일을 만들었다가 즉시 삭제합니다.


In [ ]:
import tempfile

try:
    with tempfile.NamedTemporaryFile(prefix=".ksc_write_test_", dir=REPO_ROOT) as handle:
        handle.write(b"KSC2026 write check")
        handle.flush()
    WRITE_READY = True
    print("PASS  Participant workspace is writable")
except OSError as exc:
    WRITE_READY = False
    print(f"FAIL  Participant workspace is not writable: {exc}")

usage = shutil.disk_usage(REPO_ROOT)
print(f"INFO  Free disk space: {usage.free / 2**30:.1f} GiB")

COMMON_IMAGE_READY = (
    ARCH_READY
    and CUDA_READY
    and CUDA_TENSOR_READY
    and GH200_READY
    and SM90_READY
    and GPU_COUNT_READY
    and NVIDIA_SMI_COMMAND_READY
    and DRIVER_READY
    and MANIFEST_READY
)
READY_CPU = SESSION_FILES_READY["01-1 Grace CPU"] and COMMON_IMAGE_READY and TOOLS_READY and WRITE_READY
READY_GPU = SESSION_FILES_READY["01-2 Hopper GPU"] and COMMON_IMAGE_READY and TOOLS_READY and WRITE_READY
READY_PINN = SESSION_FILES_READY["02-1 발사체 운동 PINN"] and COMMON_IMAGE_READY and STACK_READY and WRITE_READY
READY_FNO = SESSION_FILES_READY["02-2 Poisson FNO"] and COMMON_IMAGE_READY and STACK_READY and WRITE_READY

print("\n" + "=" * 72)
print(f"01-1 Grace CPU compile/tune     : {'READY' if READY_CPU else 'CHECK REQUIRED'}")
print(f"01-2 Hopper GPU memory/profile  : {'READY' if READY_GPU else 'CHECK REQUIRED'}")
print(f"02-1 PhysicsNeMo Projectile PINN: {'READY' if READY_PINN else 'CHECK REQUIRED'}")
print(f"02-2 PhysicsNeMo Poisson FNO    : {'READY' if READY_FNO else 'CHECK REQUIRED'}")
print("=" * 72)


## 판정 기준과 다음 단계

- `FAIL`이 하나라도 보이면 패키지를 설치하거나 설정을 바꾸지 말고, **실패한 셀의 전체 출력**을 진행자에게 전달합니다.
- `Visible GPUs`는 배정 정보가 전달된 세션에서는 그 수와 정확히 같아야 합니다. 배정 정보가 없는 세션의 정상 범위는 참가자용 한 개 또는 강사용 네 개입니다. 참가자에게 배정된 물리 GPU 한 개는 노트북에서 `cuda:0`으로 보입니다.
- `nvidia-smi`가 GPU를 표시하더라도 CUDA 연산이 성공했다고 단정할 수 없습니다. `CUDA tensor op`까지 `PASS`인지 확인합니다.
- 작업공간 쓰기 점검은 이후 생성할 실행 파일·보고서·체크포인트를 저장할 수 있는지 확인합니다.

네 줄이 모두 `READY`이면 환경 점검이 끝났습니다. 다음 노트북을 여세요.

## ▶ 다음: [01-1 Grace CPU 컴파일·튜닝](01_GH200/01_CPU_Compile_and_Tune.ipynb)


---

## 출처와 라이선스

이 사전 점검 노트북은 KSC 2026 통합 과정용으로 작성했습니다. 저장소 각 파일의 기존 저작권과 라이선스 고지는 그대로 적용됩니다.
